# ⚽ MLS 2024 — Football Market Intelligence (Fase 1)
## Análise de performance ofensiva ajustada por minutos (por 90)


# 📌 Fase 1 — Preparação do Dataset e Criação de Métricas Base (MLS 2024)

Nesta fase do projeto, eu executo o trabalho fundamental de um Analista de Dados: **pegar um dataset bruto**, entender sua estrutura, validar a qualidade das informações, **corrigir problemas de tipo e formato**, aplicar **regras de confiabilidade** e, só então, criar **métricas comparáveis** que permitam análises justas e consistentes.

> ✅ Resultado da Fase 1: um dataset tratado e confiável, pronto para uso em SQL, Power BI e Excel, contendo métricas por 90 minutos e um score ofensivo proprietário (versão inicial).

---

## 1) Por que a Fase 1 é necessária?

Antes de qualquer análise visual ou ranking, eu preciso garantir que:

- Os dados foram carregados corretamente
- As colunas possuem o **tipo adequado** (número vs texto)
- As métricas calculadas não estão distorcidas por problemas estruturais
- Existe uma regra mínima de confiabilidade para comparação entre jogadores

Sem essa etapa, qualquer insight posterior pode ser estatisticamente fraco ou enganoso.

---

## 2) Carregamento do dataset (RAW)

### O que eu fiz
Carreguei o arquivo CSV original em um DataFrame chamado `df_raw`.

### Por que fiz dessa forma
Eu trato `df_raw` como a **fonte original do dado**, sem aplicar transformações.  
Essa separação garante rastreabilidade e segurança caso eu precise revisar ou refazer etapas de limpeza.

---

## 3) Diagnóstico inicial do dataset

### O que eu analisei
Utilizei:
- `shape` para entender o tamanho do dataset
- `info()` para verificar tipos das colunas e valores nulos

### O que identifiquei
- O dataset possui **819 jogadores** e **16 colunas**
- A maioria das colunas já estava em formato numérico
- Algumas colunas apresentavam valores ausentes
- A coluna `minutes_played` estava em formato **texto (`str`)**, o que exigia tratamento

Esse diagnóstico guiou todas as decisões de limpeza subsequentes.

---

## 4) Por que a coluna `minutes_played` precisou de limpeza?

Eu identifiquei que `minutes_played` estava como texto ao observar:

- O tipo `str` no `info()`
- Valores com separador de milhar (ex.: `"1,637"`)

Como essa coluna é essencial para cálculos de desempenho por tempo jogado, ela precisava ser convertida para um formato numérico. Caso contrário, divisões e métricas por 90 minutos seriam inválidas.

---

## 5) Criação de uma cópia para transformação

Antes de iniciar a limpeza, criei uma cópia do dataset:

- `df_raw` → dataset original
- `df` → dataset para transformação

Essa prática me permite trabalhar com segurança, mantendo o dado original intacto.

---

## 6) Limpeza e conversão de `minutes_played`

### Objetivo
Transformar a coluna `minutes_played` de texto para número.

### Etapas aplicadas
1. Garantir que a coluna estivesse em formato string
2. Remover o separador de milhar (vírgula)
3. Converter o resultado para tipo numérico

Ao final dessa etapa, validei que a coluna estava em formato numérico (`int` ou `float`), tornando-a apta para cálculos.

---

## 7) Normalização das colunas `age` e `born`

Converti as colunas `age` e `born` para formato numérico para:

- Garantir consistência
- Evitar problemas em filtros ou análises futuras
- Tratar corretamente valores ausentes (`NaN`)

---

## 8) Análise estatística para definição de regras

Utilizei estatística descritiva (`describe`) para entender a distribuição de:

- minutos jogados
- idade
- gols
- assistências

Essa análise revelou a presença de jogadores com **pouquíssimos minutos**, o que reforçou a necessidade de um filtro de confiabilidade.

---

## 9) Definição de regra profissional: minutos mínimos

Para evitar distorções em métricas por 90 minutos, defini um critério mínimo:

- **900 minutos jogados** (aproximadamente 10 jogos completos)

Esse corte reduz o impacto de amostras pequenas e torna as comparações mais justas e confiáveis.

---

## 10) Feature Engineering: métricas por 90 minutos

Para padronizar a comparação entre jogadores com diferentes tempos em campo, criei métricas ajustadas por 90 minutos:

- `goals_per90`
- `assists_per90`
- `g_a_per90` (gols + assistências por 90)

Essas métricas não existiam no dataset original e representam um passo central da análise.

---

## 11) Criação de score ofensivo proprietário (v1)

Desenvolvi um score ofensivo inicial para organizar e priorizar jogadores com base em contribuição ofensiva.

### Definição do score
- 60% gols por 90
- 40% assistências por 90

Esse score foi nomeado como `offensive_score_v1` para indicar que se trata de uma **primeira versão**, passível de evolução nas próximas fases do projeto.

---

## 12) Validação inicial dos resultados

Após criar o score, gerei um ranking dos jogadores com melhor desempenho ofensivo para validar se os resultados eram plausíveis.

Essa validação é essencial para garantir que:
- a limpeza foi correta
- o filtro de minutos está funcionando
- as métricas fazem sentido no contexto do futebol

---

## ✅ Entregável final da Fase 1

Ao final desta fase, obtive:

- Um dataset limpo e consistente
- Colunas numéricas corretamente tipadas
- Regras de confiabilidade aplicadas
- Métricas padronizadas por 90 minutos
- Um score ofensivo proprietário (v1)

Esse dataset está pronto para ser utilizado em:
- SQL
- Power BI
- Excel
- análises estratégicas mais avançadas

---

## Próxima etapa

Na **Fase 2**, o foco será aprofundar a análise, com:
- scores específicos por posição
- identificação de perfis subvalorizados
- análises por idade, clube e contexto
- preparação para visualização executiva em BI

---

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

## 1) Carregamento do dataset (RAW)

Aqui eu importo o CSV original **sem alterações**, criando um DataFrame chamado `df_raw`.

In [2]:
df_raw = pd.read_csv("../data/raw/mls_2024_player_stats.csv")
df_raw.head()

,id,name,country,position,second_position,club,age,born,matches_played,matches_started,minutes_played,goals,assists,goals_and_assists,yellow_cards,red_cards
0,1,Liel Abada,ISR,Forward,,Charlotte,22.0,2001.0,24,20,"1,637",7,1,8,2,0
1,2,Jose Casas de Abadal,USA,Forward,Defender,Inter Miami,23.0,2000.0,2,0,31,0,0,0,0,0
2,3,Luis Abram,PER,Defender,NaN,Atlanta Utd,27.0,1996.0,19,11,"1,102",0,0,0,2,0
3,4,Lalas Abubakar,GHA,Defender,NaN,Colorado Rapids,29.0,1994.0,18,15,"1,278",0,1,1,3,0
4,5,Kellyn Acosta,USA,Midfielder,NaN,Chicago Fire,28.0,1995.0,34,27,"2,302",3,2,5,6,0


## 2) Diagnóstico do dataset

Nesta etapa eu verifico:
- número de linhas e colunas
- tipos de dados (números vs texto)
- presença de valores nulos

Isso define quais limpezas serão necessárias antes de calcular métricas.


In [3]:
df_raw.shape
df_raw.info()
df = df_raw.copy()

<class 'pandas.DataFrame'>
RangeIndex: 819 entries, 0 to 818
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 819 non-null    int64  
 1   name               819 non-null    str    
 2   country            817 non-null    str    
 3   position           819 non-null    str    
 4   second_position    257 non-null    str    
 5   club               819 non-null    str    
 6   age                817 non-null    float64
 7   born               817 non-null    float64
 8   matches_played     819 non-null    int64  
 9   matches_started    819 non-null    int64  
 10  minutes_played     819 non-null    str    
 11  goals              819 non-null    int64  
 12  assists            819 non-null    int64  
 13  goals_and_assists  819 non-null    int64  
 14  yellow_cards       819 non-null    int64  
 15  red_cards          819 non-null    int64  
dtypes: float64(2), int64(8), str(6)
memor

## 3) Limpeza crítica: `minutes_played`

`minutes_played` normalmente vem como texto quando contém separador de milhar (ex.: `"1,637"`).
Para calcular métricas por 90, precisamos transformar isso em número.

Passos:
1. garantir que é string
2. remover vírgulas
3. converter para número (`to_numeric`)

In [4]:
df["minutes_played"] = df["minutes_played"].astype(str)  # garante string
df["minutes_played"] = df["minutes_played"].str.replace(",", "", regex=False)  # remove vírgula
df["minutes_played"] = pd.to_numeric(df["minutes_played"], errors="coerce")  # vira número

In [5]:
df["minutes_played"].dtype  # deve virar int64/float64
df["minutes_played"].isna().sum()  # quantos viraram NaN (ideal: 0)

np.int64(0)

## 4) Estatística descritiva de minutos

Vou olhar mínimo, média, máximo e quartis para decidir o filtro de minutos mínimos.

In [6]:
df["minutes_played"].describe()

count     819.000000
mean     1189.273504
std       911.738005
min         1.000000
25%       359.000000
50%      1041.000000
75%      2001.500000
max      3060.000000
Name: minutes_played, dtype: float64

## 5) Filtro de minutos mínimos

Vou filtrar jogadores com amostra pequena para não distorcer métricas por 90.

Critério inicial:
- 900 minutos (≈ 10 jogos completos)

Depois ajustamos se necessário.

In [7]:
MIN_MINUTES = 900  # critério
df = df[df["minutes_played"] >= MIN_MINUTES].copy()  # aplica filtro
df.shape  # ver quantos sobraram

(444, 16)

## 6) Métricas por 90

Vou padronizar performance por 90 minutos:
- gols por 90
- assistências por 90
- (gols + assistências) por 90

In [8]:
df["goals_per90"] = (df["goals"] / df["minutes_played"]) * 90
df["assists_per90"] = (df["assists"] / df["minutes_played"]) * 90
df["g_a_per90"] = ((df["goals"] + df["assists"]) / df["minutes_played"]) * 90

## 7) Score ofensivo (v1)

Vou criar um score simples:
- 60% gols por 90
- 40% assistências por 90

In [9]:
df["offensive_score_v1"] = (df["goals_per90"] * 0.6) + (df["assists_per90"] * 0.4)

## 8) Top 10 (validação rápida)

Se o top 10 fizer sentido, seguimos para análises por posição.

In [10]:
df.sort_values("offensive_score_v1", ascending=False)[
    ["name", "club", "position", "minutes_played", "goals", "assists", "offensive_score_v1"]
].head(10)

,name,club,position,minutes_played,goals,assists,offensive_score_v1
476,Lionel Messi,Inter Miami,Forward,1489,20,11,0.991269
712,Luis Suárez,Inter Miami,Forward,1918,20,9,0.732013
182,Cucho,Columbus Crew,Forward,2113,19,10,0.655939
454,Alonso Martínez,NYCFC,Forward,1488,16,3,0.653226
544,Tani Oluwaseyi,Minnesota Utd,Forward,1084,8,5,0.564576
238,Evander,Portland Timbers,Midfielder,2459,15,15,0.549004
85,Christian Benteke,D.C. United,Forward,2594,23,5,0.548188
298,Carlos Gómez,Real Salt Lake,Forward,1807,13,7,0.527947
370,Dejan Joveljić,LA Galaxy,Forward,1927,15,5,0.513752
456,Josef Martínez,CF Montréal,Forward,1376,11,3,0.510174
